# Interpretability


Every lecture since 13 has judged a model from the outside: a benchmark score,
an arena rating, a trajectory grade. This last lecture before the quiz turns
the model inside out: not "how well does it do?" but "what is it computing,
and can we read that off the weights and activations?" The objects of study
are ones you already own: the gradient from lecture 05, the residual stream
from lecture 08, the attention heads and MLPs from lectures 10 and 11, and an
L1 penalty (lecture 06 covered the L2 case; L1 drives coefficients exactly to
zero, as section 4 derives). UDL does not cover this material; the notation
follows the primary papers, with UDL's $\boldsymbol{\Omega}$ for weight
matrices and $\mathbf{h}_l$ for the residual stream kept throughout. The three
calculations to carry out are integrated gradients on a saturating function,
a normalised activation-patching effect from three logit differences, and the
soft-threshold solution that explains why an L1 penalty produces exactly-zero
features.


## Today's roadmap

1. Why look inside: what evals cannot tell you; attribution vs. mechanistic interpretability; integrated gradients
2. The residual stream as the object of study: heads as readers and writers, the logit lens, induction heads
3. Linear representations and probing; activation patching as the causal test
4. Superposition and sparse autoencoders
5. Circuit tracing and attribution graphs
6. Steering, editing and safety audits
7. Honest limits: faithfulness, the gap between features and understanding, tools you can run
8. Summary


# Why look inside

Lectures 13 and 22 built evaluations: benchmarks, judges, arenas, agent task
suites. This section says what those cannot tell you, and introduces the two
traditions that try to answer it by reading the model itself.


## Evals say *what*, not *why*

- A benchmark score samples behaviour on one input distribution. It cannot
  say **why** the answer was right (general mechanism, or memorised
  shortcut?), whether it **transfers** off-distribution, or whether a failure
  you have not observed is **lurking** (a backdoor, a hidden objective, a
  spurious cue). Lecture 13's contamination problem is the special case where
  the hidden reason is "it saw the test set".
- Three reasons to open the box:
  1. **Debugging.** Which input feature drove this prediction? Which layer
     stores the fact that is wrong?
  2. **Safety.** Does the model say what it is doing? Can a goal it was not
     supposed to have be detected from the inside, before it acts?
  3. **Science.** What algorithms does gradient descent find, and are the
     representations human-meaningful?
- The state of the field in 2026: partial answers to all three, with tools
  that run on open-weight models on one GPU.


## Two traditions

| | **Attribution / saliency** | **Mechanistic interpretability** |
|---|---|---|
| Question | Which *inputs* mattered for this output? | What *computation* produced it, in terms of internal parts? |
| Unit | pixel, token, input feature | neuron, attention head, direction, feature, circuit |
| Tool | gradients, path integrals, perturbations | residual-stream algebra, probes, patching, dictionary learning |
| Output | a heat-map over the input | a mechanism, tested by intervention |
| Era | 2013 onward (CNN classifiers) | 2020 onward (transformers) |
| Papers | [Simonyan et al. 2013](https://arxiv.org/abs/1312.6034), [Sundararajan et al. 2017](https://arxiv.org/abs/1703.01365), [Selvaraju et al. 2017](https://arxiv.org/abs/1610.02391) | [Elhage et al. 2021](https://transformer-circuits.pub/2021/framework/index.html), [Olsson et al. 2022](https://arxiv.org/abs/2209.11895), [Bricken et al. 2023](https://transformer-circuits.pub/2023/monosemantic-features/index.html) |

Attribution is the older, cheaper tradition and is still what most "explain
this prediction" buttons do: three slides. The rest of the lecture is about
mechanisms, where the 2024–26 progress is.


## Gradient saliency, and why it is not enough

The simplest attribution for a class score $F[\mathbf{x}]$ is the gradient
([Simonyan et al. 2013](https://arxiv.org/abs/1312.6034)):
$$
\mathrm{saliency}_i = \left|\frac{\partial F[\mathbf{x}]}{\partial x_i}\right|
\qquad\text{or}\qquad
\mathrm{grad}\times\mathrm{input}_i = x_i \,\frac{\partial F[\mathbf{x}]}{\partial x_i}.
$$

- One backward pass (lecture 05), so it is free. It answers "if I nudged this
  pixel, how much would the score move?", a *local* question.
- **Saturation.** If the response to a feature has saturated (a ReLU that is
  off, a flat sigmoid), the gradient at the input is zero even though the
  feature was decisive. The next figure makes this concrete.
- **Sanity checks fail.** [Adebayo et al. 2018](https://arxiv.org/abs/1810.03292):
  several popular saliency maps barely change when the weights are randomised
  or the labels shuffled; they were edge detectors on the input, not
  explanations of the model.
- So we want an attribution tied to the function by axioms, not plausibility.


## Integrated gradients: the axioms and the formula

[Sundararajan, Taly and Yan (2017)](https://arxiv.org/abs/1703.01365) ask what
an attribution *must* satisfy and show that integrated gradients is the unique
path method that also preserves symmetry (Sundararajan et al. §4).

- **Sensitivity.** If one input differs from a baseline $\mathbf{x}'$ and the
  output differs, that input gets non-zero attribution (the plain gradient
  violates this under saturation).
- **Implementation invariance.** Two networks computing the same function
  get the same attributions.
- **Integrated gradients** accumulate the gradient along the straight line
  from the baseline (a black image, an all-zero embedding) to the input:
  $$
  \mathrm{IG}_i[\mathbf{x}] = (x_i - x_i') \int_0^1
  \frac{\partial F[\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}')]}{\partial x_i}\, d\alpha
  \;\approx\; (x_i - x_i') \,\frac{1}{m}\sum_{k=1}^{m}
  \frac{\partial F[\mathbf{x}' + \tfrac{k}{m}(\mathbf{x} - \mathbf{x}')]}{\partial x_i}.
  $$
- **Completeness** (the fundamental theorem of calculus):
  $\sum_i \mathrm{IG}_i[\mathbf{x}] = F[\mathbf{x}] - F[\mathbf{x}']$. The
  attributions *add up to the score change*, so they can be read as shares.
- Cost: $m$ backward passes, $m$ from 20 to 300 in the paper.


## Integrated gradients: a worked example

![A saturating one-dimensional output. The gradient at the input is zero, so gradient attribution calls the input irrelevant; integrating the gradient along the path from the baseline recovers the full effect, and completeness holds. Source: instructor figure.](figs/23-Interpretability/integrated-gradients-saturation.png)

**Quizzable.** $F[x] = \min(x, 1)$, input $x = 3$, baseline $x' = 0$. The
gradient at the input is $0$. Along the path, $F'[3\alpha] = 1$ for
$\alpha < 1/3$ and $0$ after, so
$\mathrm{IG} = (3 - 0)\int_0^1 F'[3\alpha]\,d\alpha = 3 \times \tfrac13 = 1 = F[3] - F[0]$.
A second one: $F[x_1, x_2] = x_1 x_2$ with baseline $\mathbf{0}$ gives
$\mathrm{IG}_1 = x_1 \int_0^1 \alpha x_2\, d\alpha = \tfrac12 x_1 x_2$, likewise
$\mathrm{IG}_2$, summing to $F$; gradient$\times$input gives $x_1 x_2$ to each
and sums to $2F$, violating completeness.


Grad-CAM ([Selvaraju et al. 2017](https://arxiv.org/abs/1610.02391)) is the
CNN-era analogue. **What attribution cannot do.** A heat-map says *where* the
evidence was, not *how* the network combined it: not that the model
implements a lookup table for addition, or plans a rhyme a line ahead. For
that we look at the internal parts, and in a transformer the place to look is
the residual stream.


Both worked examples, with the integral replaced by the lecture's Riemann sum
over $m$ steps. The plain gradient calls the saturated input irrelevant;
integrated gradients converge to the closed forms ($1$, and
$\tfrac12 x_1 x_2$ each) and their sum to $F[\mathbf{x}] - F[\mathbf{x}']$ as
$m$ grows, while gradient$\times$input on $x_1 x_2$ sums to $2F$. The
right-endpoint sum is only first-order accurate, so the error falls as $1/m$
(unevenly at the kink of $\min(x, 1)$, as at a ReLU), which is why the paper
uses up to 300 steps and checks completeness to choose $m$.

In [ ]:
import numpy as np

def integrated_gradients(grad_F, x, baseline, m):
    """The lecture's Riemann sum: m gradients at k/m of the way from baseline to x."""
    alphas = np.arange(1, m + 1) / m
    grads = np.array([grad_F(baseline + a * (x - baseline)) for a in alphas])
    return (x - baseline) * grads.mean(axis=0)

# Example 1: F[x] = min(x, 1), input x = 3, baseline x' = 0.
F1 = lambda x: np.minimum(x, 1.0)
grad_F1 = lambda x: (x < 1.0).astype(float)
x1, base1 = np.array([3.0]), np.array([0.0])

# Example 2: F[x1, x2] = x1 * x2, baseline 0.
F2 = lambda x: x[0] * x[1]
grad_F2 = lambda x: np.array([x[1], x[0]])
x2, base2 = np.array([2.0, 3.0]), np.zeros(2)

print("F = min(x, 1), x = 3, x' = 0:  gradient at the input =", grad_F1(x1)[0])
print("F = x1 x2, x = (2, 3), x' = 0: gradient x input =", x2 * grad_F2(x2),
      "sums to", (x2 * grad_F2(x2)).sum(), "= 2F")
print(f"\n{'m':>5} {'IG, min(x,1)':>13} {'IG_1, x1x2':>11} {'IG_2, x1x2':>11} {'sum IG':>7}")
for m in (5, 20, 100, 300, 3000):
    ig1 = integrated_gradients(grad_F1, x1, base1, m)
    ig2 = integrated_gradients(grad_F2, x2, base2, m)
    print(f"{m:>5} {ig1[0]:>13.4f} {ig2[0]:>11.4f} {ig2[1]:>11.4f} {ig2.sum():>7.4f}")
print(f"{'exact':>5} {1.0:>13.4f} {F2(x2) / 2:>11.4f} {F2(x2) / 2:>11.4f} {F2(x2):>7.4f}")
print(f"\nCompleteness targets: F[3] - F[0] = {(F1(x1) - F1(base1))[0]:.1f}, "
      f"F[x] - F[0] = {F2(x2) - F2(base2):.1f}")

# The residual stream

Lecture 08 ended with the observation that a pre-norm transformer's residual
stream is an un-normalised sum of every block's contribution. Mechanistic
interpretability takes that sum literally: it is a shared communication
channel, and each head and MLP is a small program that reads from it and writes
to it. This section sets up that picture, following
[Elhage et al. (2021)](https://transformer-circuits.pub/2021/framework/index.html).


## The transformer block, seen as a bus

![The residual stream as a shared channel. Every attention head and MLP reads the stream (through a norm and its own input weights) and adds a vector back; the stream at any depth can be read with the final unembedding, the logit lens. Source: instructor figure, after Elhage et al. (2021).](figs/23-Interpretability/residual-stream-read-write.png)

- Lecture 08's identity, per token position:
  $\mathbf{h}_L = \mathbf{h}_0 + \sum_{l}\big(\mathbf{a}_l + \mathbf{m}_l\big)$,
  the embedding plus every head's and MLP's output, added, never overwritten.
  Logits are $\boldsymbol{\Omega}_U\,\mathrm{Norm}[\mathbf{h}_L]$.
- Consequences ([Elhage et al. 2021](https://transformer-circuits.pub/2021/framework/index.html)):
  the stream has **no privileged basis** (rotating all weights gives the same
  model, so "dimension 437 of the stream" means nothing); components talk
  only through it, so a layer-3 head can **read what a layer-1 head wrote**;
  and its $D$ dimensions are shared by thousands of components, which forces
  the superposition of section 4.
- The question becomes: which *directions* carry which information, and
  which components write and read them?


## Attention heads read and write: the QK and OV circuits

Drop the biases from lecture 10 and write head $h$'s contribution at position
$n$, with $\boldsymbol{\Omega}_o^h$ head $h$'s slice of UDL's combining matrix
$\boldsymbol{\Omega}_c$:
$$
\mathbf{a}^h_n = \boldsymbol{\Omega}_o^h \sum_m \mathrm{a}_h[\mathbf{x}_m, \mathbf{x}_n]\,\boldsymbol{\Omega}_v^h \mathbf{x}_m
= \sum_m \mathrm{a}_h[\mathbf{x}_m, \mathbf{x}_n]\;\underbrace{\big(\boldsymbol{\Omega}_o^h \boldsymbol{\Omega}_v^h\big)}_{\text{OV circuit},\ D\times D}\mathbf{x}_m,
$$
$$
\mathrm{a}_h[\mathbf{x}_m, \mathbf{x}_n] = \mathrm{softmax}_m\Big[\mathbf{x}_m^{\mathsf T}\underbrace{\big(\boldsymbol{\Omega}_k^{h\mathsf T}\boldsymbol{\Omega}_q^h\big)}_{\text{QK circuit},\ D\times D}\mathbf{x}_n\Big].
$$

- The head factors into two independent low-rank (rank $\le D/H$) maps: the
  **QK circuit** decides *where to attend*, the **OV circuit** *what to copy*
  from the attended position into the stream. Elhage et al. write
  $W_{OV} = W_O W_V$ and $W_{QK} = W_Q^{\mathsf T} W_K$ (our $D \times D$
  matrix has the key on the left, matching lecture 10; Elhage et al. put the
  query on the left, so theirs is the transpose).
- **Virtual weights.** Since everything is added into one stream, one head's
  OV matrix times a later head's QK or OV matrix is a direct linear path
  between them: heads compose through the query, key or value.
- The **direct path** $\boldsymbol{\Omega}_U\boldsymbol{\Omega}_E$, with
  $\boldsymbol{\Omega}_E$ the embedding matrix, is a bigram table; one
  attention layer adds "skip-trigrams" (`A ... B → C`);
  two layers is where something new appears.
- Exact for attention-only models; the MLPs' non-linearities are what the
  dictionary methods of section 4 exist to handle.


## The logit lens

[nostalgebraist (2020)](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)
noticed that since every layer only *adds* to the stream, you can apply the
final norm and unembedding at *any* depth and ask what the model would predict
if it stopped here:
$$
\mathrm{Pr}_l(y \mid \mathbf{x}) = \mathrm{softmax}\big[\boldsymbol{\Omega}_U\,\mathrm{Norm}[\mathbf{h}_l]\big].
$$

- In GPT-2 the top-1 token under the lens often converges to the final answer
  well before the last layer, and the rank of the correct token falls
  steadily with depth: later layers refine a guess already present.
- Descriptive, free, and the first thing to try on an open-weight model. It
  fails instructively: early layers of some models are unreadable because the
  stream is not yet in the "output basis".
  [Belrose et al. (2023)](https://arxiv.org/abs/2303.08112) fix this with the
  **tuned lens**, a small learned affine map per layer trained to match the
  final distribution.
- The lens is the simplest **probe** (section 3): a linear read-out of the
  stream with a fixed, meaningful matrix.


## Induction heads: the first well-understood circuit

![An induction head on a repeated random sequence. At the second occurrence of a token the head attends to the token that followed its first occurrence, and copies it, raising that token's logit. Figure from Olsson et al. (2022), "In-context Learning and Induction Heads", arXiv:2209.11895.](figs/23-Interpretability/induction-head-olsson.png)

- Definition ([Olsson et al. 2022](https://arxiv.org/abs/2209.11895)): on a
  repeated random sequence `[A][B] ... [A]`, a head that (i) **prefix-matches**,
  attending from the second `A` to the `B` that followed the first `A`, and
  (ii) **copies**, raising the logit of `B`.
- Mechanism ([Elhage et al. 2021](https://transformer-circuits.pub/2021/framework/index.html)):
  a **previous-token head** in an earlier layer writes "the token before me
  was `A`" into `B`'s stream; the induction head's QK circuit, by
  K-composition, matches the current `A` against that key, and its OV circuit
  copies `B`. Two layers are necessary; one-layer models cannot do it.
- It generalises to `[A*][B*] ... [A] → [B]` with "similar" tokens, a form of
  in-context nearest-neighbour lookup.


## Induction heads and in-context learning

- Olsson et al. trained 34 transformers and watched them. Early in training
  there is a **phase change**, a bump in the loss curve, in which induction
  heads form and in-context learning (loss on late tokens minus loss on early
  tokens) improves abruptly, in every model with more than one layer.
- Six lines of evidence that the heads *cause* the ability: co-occurrence in
  time; architectural changes that shift when the heads can form shift the
  ability; ablating the heads removes most in-context learning in small
  models; the heads do abstract copying (translation, pattern completion);
  the mechanism plausibly supports in-context learning; and the behaviour is
  continuous from small to large models.
- A second hand-found circuit: **indirect object identification** in GPT-2
  small ([Wang et al. 2022](https://arxiv.org/abs/2211.00593)), "When Mary
  and John went to the store, John gave a drink to → Mary": 26 heads in 7
  classes (name movers, S-inhibition heads, duplicate-token heads, ...),
  found by patching one head at a time.
- Both took months of human effort for one behaviour in a tiny model. The
  rest of the lecture is about tools that scale.


# Linear representations and probing

Circuits are about components. The complementary view is about *directions*:
the hypothesis that a transformer represents a concept as a direction in the
residual stream, and that we can find it with a classifier. This section
introduces probes, the result that made them famous, what a probe can and
cannot show, and the causal test that settles it.


## The linear representation hypothesis

- Old evidence: word2vec analogies,
  $\mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}} \approx \mathbf{v}_{\text{queen}}$
  ([Mikolov et al. 2013](https://arxiv.org/abs/1301.3781)). A concept is a
  *direction*; adding it changes the concept.
- The hypothesis for LLMs: high-level concepts are linear directions in the
  residual stream, so they can be *read* with a linear probe and *written* by
  adding a vector. [Park, Choe and Veitch (2023)](https://arxiv.org/abs/2311.03658)
  make it precise and test it on Llama 2.
- Probe evidence: Llama 2 represents the **latitude and longitude** of places
  and the **time** of events linearly
  ([Gurnee and Tegmark 2023](https://arxiv.org/abs/2310.02207)); models have
  a linear **truth** direction that generalises across true/false datasets
  ([Marks and Tegmark 2023](https://arxiv.org/abs/2310.06824)).
- Why linear? The only way a later component reads the stream is through a
  linear map (a QK/OV circuit, an MLP's first layer), so linearly readable
  features are the useful ones for the model to write.


## Linear probes

[Alain and Bengio (2016)](https://arxiv.org/abs/1610.01644): freeze the model,
collect activations $\mathbf{h}_l$ with labels $y$ (a property the model was
*never trained to predict*), and fit a linear classifier:
$$
\mathrm{Pr}(y \mid \mathbf{h}_l) = \mathrm{softmax}\big[\mathbf{W}\mathbf{h}_l + \mathbf{b}\big],
\qquad
\text{minimise cross-entropy over } (\mathbf{W}, \mathbf{b}) \text{ only.}
$$

- Held-out accuracy, layer by layer, maps *where* a property becomes linearly
  available: syntax early, semantics and world-state in the middle,
  next-token specifics late.
- **The probe must be weaker than the question.** A two-layer MLP probe can
  compute the property *itself*; then high accuracy means the probe learned
  it, not the model. [Hewitt and Liang (2019)](https://arxiv.org/abs/1909.03368)
  add a **control task** with random labels of the same shape and report
  *selectivity*, task accuracy minus control accuracy.
- [Belinkov (2022)](https://arxiv.org/abs/2102.12452) surveys the pitfalls.
  The deepest: a probe shows information is **present**, not that the model
  **uses** it.


A probe on synthetic "activations": a binary property written along one
direction $\mathbf{w}$ of a 64-dimensional stream, plus unit Gaussian noise.
The probe is fitted in closed form (ridge-regularised least squares on targets
$\pm 1$, the simplest linear classifier) on 200 examples. The control task
fits the same probe to random labels: it scores well above chance on its own
training set, because 64 dimensions can memorise 200 labels, and at chance on
held-out data. Only held-out accuracy, and the gap to the control, measure
what the activations contain. Lower `signal` towards zero and watch the
selectivity vanish.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
D, n_train, n_test = 64, 200, 2000
signal = 1.5          # how strongly the property is written along w_true
noise = 1.0

w_true = rng.standard_normal(D)
w_true /= np.linalg.norm(w_true)

def activations(y):
    return signal * (2 * y[:, None] - 1) * w_true + noise * rng.standard_normal((len(y), D))

y_train, y_test = rng.integers(0, 2, n_train), rng.integers(0, 2, n_test)
H_train, H_test = activations(y_train), activations(y_test)

def with_bias(H):
    return np.hstack([H, np.ones((len(H), 1))])

def fit_probe(H, y, ridge=1.0):
    """Least-squares linear probe on targets +-1, with a bias (closed form)."""
    X = with_bias(H)
    return np.linalg.solve(X.T @ X + ridge * np.eye(X.shape[1]), X.T @ (2 * y - 1))

def accuracy(w, H, y):
    return np.mean((with_bias(H) @ w > 0) == y)

w = fit_probe(H_train, y_train)
y_ctrl_train, y_ctrl_test = rng.integers(0, 2, n_train), rng.integers(0, 2, n_test)
w_ctrl = fit_probe(H_train, y_ctrl_train)            # control task: random labels

task = accuracy(w, H_train, y_train), accuracy(w, H_test, y_test)
ctrl = accuracy(w_ctrl, H_train, y_ctrl_train), accuracy(w_ctrl, H_test, y_ctrl_test)
print(f"{'':>15} {'train acc':>10} {'test acc':>9}")
print(f"{'property':>15} {task[0]:>10.3f} {task[1]:>9.3f}")
print(f"{'random labels':>15} {ctrl[0]:>10.3f} {ctrl[1]:>9.3f}")
print(f"selectivity (test accuracy, task - control): {task[1] - ctrl[1]:.3f}")
cos = w[:D] @ w_true / np.linalg.norm(w[:D])
print(f"cosine between the probe's direction and w_true: {cos:.2f}")

## Othello-GPT: a world model you can probe

- [Li et al. (2023)](https://arxiv.org/abs/2210.13382) train a GPT on
  sequences of *legal Othello moves* (synthetic games; tokens are squares).
  It never sees a board, and learns to predict legal moves almost perfectly.
- Does it represent the board? **Nonlinear** (two-layer MLP) probes recover
  every square's state with low error; **linear** probes do much worse.
  Li et al. Table 1: linear probes reach 20.4–23.1% error (best layer 3) vs
  26.7–28.9% on a randomised network; 2-layer MLP probes reach 1.7%
  (synthetic) / 9.4% (championship); a constant "empty" guess scores 52.95%
  error.
  Editing the probe-recovered board (flipping a square through the probe's
  gradient) changes which moves the model calls legal: the representation is
  *used*.
- The twist: [Nanda, Lee and Wattenberg (2023)](https://arxiv.org/abs/2309.00941)
  showed the board **is** linear once you ask the right question. The model
  encodes not "black / white" but "**mine / theirs**" relative to the player
  to move, which flips every turn. In that basis a linear probe works, and
  adding the probe direction to the stream causally flips a square.
- Lesson: a negative linear result may mean you chose the wrong variable,
  not that the representation is nonlinear.


## What a probe can and cannot show

- **Can show:** the information is linearly decodable at layer $l$; where in
  depth it appears; which direction $\mathbf{w}$ carries it (a candidate for
  steering in section 6).
- **Cannot show:** that any downstream component reads $\mathbf{w}$. The
  stream carries more than the model uses at any position; a probe can succeed
  on information that is incidental, redundant, or left over from the input.
- **Cannot show:** that the model carves the concept the way *you* labelled
  it (the mine/theirs lesson), or that the probe's direction is the model's
  direction rather than the one that separated your dataset.
- The repair is always the same: **intervene**. Move the activation along
  the direction and watch the output; if nothing happens, the model was not
  using it. This is the methodological spine of the field: *correlation from
  observation, causation from intervention*.


## Activation patching: the causal test

![Left: a probe is a classifier on frozen activations and is purely observational. Right: activation patching copies one activation from a clean run into a corrupted run and measures how much of the clean output is restored. Source: instructor figure.](figs/23-Interpretability/probe-vs-patching.png)

- Two prompts differing in one fact: **clean** "The capital of the state
  containing Dallas is" (→ Austin) and **corrupted** "... containing Oakland
  is" (→ Sacramento). Run both, cache every activation. Run the corrupted
  prompt again but **overwrite one activation** (a layer, a position, a head)
  with its clean value, and measure the output.
- Metric: the **logit difference** $\Delta = \mathrm{logit}(\text{Austin}) - \mathrm{logit}(\text{Sacramento})$
  at the last position, more stable than a probability
  ([Zhang and Nanda 2023](https://arxiv.org/abs/2309.16042)). Normalise:
  $$
  \text{effect} = \frac{\Delta_{\text{patched}} - \Delta_{\text{corrupted}}}{\Delta_{\text{clean}} - \Delta_{\text{corrupted}}},
  \qquad 0 = \text{no restoration},\ 1 = \text{full restoration}.
  $$
- **Quizzable.** $\Delta_{\text{clean}} = 4.0$, $\Delta_{\text{corrupted}} = -1.0$,
  patching layer 12's stream at the city position gives
  $\Delta_{\text{patched}} = 3.0$: effect $= (3.0 + 1.0)/(4.0 + 1.0) = 0.8$.
  That one activation carries 80% of what separates the two answers.
- **Direction matters.** Clean into corrupted ("denoising") asks whether the
  activation is *sufficient* to restore the behaviour; corrupted into clean
  ("noising") asks whether it is *necessary*; redundant paths pass one and
  fail the other ([Heimersheim and Nanda 2024](https://arxiv.org/abs/2404.15255)).
- **Attribution patching** ([Nanda 2023](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching))
  replaces a sweep over layers and positions by the first-order estimate
  $\Delta \approx (\mathbf{h}_{\text{clean}} - \mathbf{h}_{\text{corrupted}})^{\mathsf T}\, \partial \Delta/\partial \mathbf{h}$:
  one forward and one backward pass for every site at once.


The normalised effect as a function, with the quizzable numbers and a few
other patched values to show the scale: $0$ when the patch changes nothing,
$1$ at full restoration, and outside $[0, 1]$ when a patch overshoots or
pushes the wrong way.

In [ ]:
def patching_effect(delta_patched, delta_clean, delta_corrupted):
    """Fraction of the clean-vs-corrupted logit difference that the patch restores."""
    return (delta_patched - delta_corrupted) / (delta_clean - delta_corrupted)

delta_clean = 4.0        # logit(Austin) - logit(Sacramento), clean prompt (Dallas)
delta_corrupted = -1.0   # same, corrupted prompt (Oakland)
delta_patched = 3.0      # corrupted run with layer 12's stream at the city position patched in

print(f"effect = ({delta_patched} - ({delta_corrupted})) / ({delta_clean} - ({delta_corrupted})) "
      f"= {patching_effect(delta_patched, delta_clean, delta_corrupted):.2f}")
for d in (-1.0, 0.0, 1.5, 4.0, 5.0, -2.0):
    print(f"  Delta_patched = {d:+.1f}: effect {patching_effect(d, delta_clean, delta_corrupted):+.2f}")

## Causal tracing, and where facts live

![Average causal-tracing effects over 1,000 factual prompts in GPT-2 XL: restoring a single hidden state (left), an MLP output (middle) or an attention output (right) at each token and layer after the subject embeddings were corrupted with noise. Figure 2 of Meng et al. (2022), "Locating and Editing Factual Associations in GPT", arXiv:2202.05262.](figs/23-Interpretability/causal-tracing-meng-fig2.png)

- [Meng et al. (2022)](https://arxiv.org/abs/2202.05262) patch clean into
  corrupted (the "denoising" direction above, a test of sufficiency):
  corrupt the **subject** tokens' embeddings with Gaussian noise so the
  model forgets ("The Space Needle is in → ?"), then copy one clean hidden
  state at a time into the corrupted run and measure recovery.
- Over 1,000 facts in GPT-2 XL, two sites matter: **MLPs in the early-middle
  layers at the last subject token**, and **attention at the last token in
  late layers**. Their reading: mid-layer MLPs are a key-value store that
  *retrieves* the fact once the subject is complete
  ([Geva et al. 2021](https://arxiv.org/abs/2012.14913)); late attention
  *moves* it to the output position.
- This localisation motivated the editing method ROME, and the dispute over
  whether localisation implies editability (both in section 6).


# Superposition and sparse autoencoders

Probes need you to name the concept in advance. The methods in this section
try to discover the concepts: they assume the model packs more features than
it has dimensions, and they learn an over-complete dictionary that unpacks
them. An L1 penalty (lecture 06 covered the L2 case; L1 drives coefficients
exactly to zero, as the "Why L1 gives sparsity" slide derives) is the engine.


## Polysemantic neurons

- The hope, from CNN work ([Olah et al. 2020](https://distill.pub/2020/circuits/zoom-in/)),
  was that neurons would be **monosemantic**: one neuron, one concept (a
  curve detector, a dog's snout). Some are.
- In language models most MLP neurons are **polysemantic**: one neuron in a
  small transformer fires on academic citations, English dialogue, HTTP
  requests and Korean text ([Bricken et al. 2023](https://transformer-circuits.pub/2023/monosemantic-features/index.html)).
  Reading neurons does not scale, because the units are not the concepts.
- Two possible reasons: the concepts are not aligned with the neuron basis (a
  rotation would fix it), or there are **more concepts than neurons** and no
  basis separates them. The toy-models paper argues for the second and names
  it **superposition**.


## The superposition hypothesis

![As feature sparsity increases, a toy model trained to reconstruct five features through a two-dimensional bottleneck goes from storing only the two most important features orthogonally, to antipodal pairs, to all five as a pentagon with some interference. Figure from Elhage et al. (2022), "Toy Models of Superposition", arXiv:2209.10652.](figs/23-Interpretability/superposition-toy-models-elhage.png)

- [Elhage et al. (2022)](https://arxiv.org/abs/2209.10652) train a tiny
  model, $\mathbf{x} \in \mathbb{R}^{5} \to \mathbf{h} = \mathbf{W}\mathbf{x} \in \mathbb{R}^{2} \to \hat{\mathbf{x}} = \mathrm{ReLU}[\mathbf{W}^{\mathsf T}\mathbf{h} + \mathbf{b}]$,
  on data whose features are **sparse** (each zero with probability $S$).
- With dense features it keeps the two most important (PCA). As sparsity
  rises it stores *more features than dimensions* as nearly-orthogonal
  directions, accepting a little interference: sparse features are rarely
  active together, and the ReLU filters the small cross-talk.
- The number of almost-orthogonal directions in $D$ dimensions grows
  exponentially with $D$, so a 4096-dimensional stream could carry millions of
  sparse features: a model is a compressed simulation of a much larger,
  sparser, monosemantic model.
- If so, the way to see the features is to *decompress*: find an
  over-complete set of directions such that each activation is a sparse
  combination of them. That is dictionary learning.


The toy model's two candidate solutions, fixed by hand rather than trained:
store features 0 and 1 on orthogonal axes and drop the rest, or put all five
on the corners of a pentagon and let the ReLU (with a small negative bias,
the best of a grid) cut the cross-talk. Feature $i$ has importance $0.7^i$ and
is uniform on $[0, 1]$ when active. The off-diagonal entries of
$\mathbf{W}^{\mathsf T}\mathbf{W}$ are the interference; it only costs
anything when two features are active at once, and the last column shows how
fast that becomes rare as the sparsity $S$ rises. Dense features favour the
orthogonal pair; sparse features favour superposition, the transition in
Elhage et al.'s figure.

In [ ]:
#| code-fold: true
#| fig-cap: "Importance-weighted reconstruction loss of two hand-built toy models with five features and a two-dimensional bottleneck. The orthogonal pair wins for dense features; the pentagon in superposition wins once features are sparse."
#| fig-alt: "Log-scale line chart of loss against sparsity S from 0 to 0.99. Both curves fall as S rises. At S equal to 0 the orthogonal model's loss (about 0.36) is below the pentagon's (about 0.47); the curves cross between S equal to 0.3 and 0.5, and at S equal to 0.99 the pentagon's loss is about four times smaller."
import numpy as np
import matplotlib.pyplot as plt

n_feat = 5
importance = 0.7 ** np.arange(n_feat)          # feature 0 matters most
angles = 2 * np.pi * np.arange(n_feat) / n_feat
W_pent = np.vstack([np.cos(angles), np.sin(angles)])   # 2 x 5: all five, as a pentagon
W_orth = np.zeros((2, n_feat))
W_orth[0, 0] = W_orth[1, 1] = 1                        # 2 x 5: features 0 and 1 only

def loss(W, x, b):
    x_hat = np.maximum(x @ W.T @ W + b, 0)             # x_hat = ReLU[W^T W x + b]
    return (importance * (x_hat - x) ** 2).sum(1).mean()

rng = np.random.default_rng(0)
n = 20_000
biases = np.linspace(-0.5, 0.0, 11)
sparsities = np.array([0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99])
loss_orth, loss_pent = [], []
for S in sparsities:
    x = rng.uniform(0, 1, (n, n_feat)) * (rng.uniform(size=(n, n_feat)) > S)
    loss_orth.append(loss(W_orth, x, 0.0))
    loss_pent.append(min(loss(W_pent, x, b) for b in biases))

print("W^T W for the pentagon (off-diagonal = cross-talk):")
print(np.round(W_pent.T @ W_pent, 2))
print(f"\n{'S':>5} {'orthogonal':>11} {'pentagon':>9} {'P(2+ features active)':>22}")
for S, lo, lp in zip(sparsities, loss_orth, loss_pent):
    p_multi = 1 - S**n_feat - n_feat * (1 - S) * S ** (n_feat - 1)
    print(f"{S:>5.2f} {lo:>11.4f} {lp:>9.4f} {p_multi:>22.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(sparsities, loss_orth, "o-", label="2 features, orthogonal (the PCA choice)")
ax.semilogy(sparsities, loss_pent, "s-", label="5 features in superposition (pentagon)")
ax.set_xlabel("sparsity $S$ (probability each feature is zero)")
ax.set_ylabel("importance-weighted loss")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Dictionary learning with a sparse autoencoder

![A sparse autoencoder. A $D$-dimensional activation is encoded into a much wider sparse code and decoded back; the decoder's columns are the dictionary of feature directions, and the loss is reconstruction plus a sparsity penalty. Source: instructor figure.](figs/23-Interpretability/sae-architecture.png)

- Collect residual-stream (or MLP) activations $\mathbf{x} \in \mathbb{R}^D$
  over billions of tokens and train, with $M \gg D$ (8× to 256× wider):
  $$
  \mathbf{f} = \mathrm{ReLU}\big[\boldsymbol{\Omega}_{\text{enc}}(\mathbf{x} - \mathbf{b}_{\text{dec}}) + \mathbf{b}_{\text{enc}}\big],
  \quad
  \hat{\mathbf{x}} = \sum_{i=1}^{M} f_i\,\mathbf{d}_i + \mathbf{b}_{\text{dec}},
  \quad
  L = \tfrac12\|\mathbf{x} - \hat{\mathbf{x}}\|_2^2 + \lambda \sum_i f_i\,\|\mathbf{d}_i\|_2 .
  $$
- Reconstruction makes the dictionary *complete*; the L1 term makes each
  code **sparse**. The factor $\|\mathbf{d}_i\|_2$ blocks the escape of
  shrinking $f_i$ while growing $\mathbf{d}_i$
  ([Templeton et al. 2024](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)).
- Variants set sparsity directly: **TopK** keeps the $k$ largest
  pre-activations ([Gao et al. 2024](https://arxiv.org/abs/2406.04093));
  **JumpReLU** learns a per-feature threshold and penalises $L_0$ through a
  straight-through estimator ([Rajamanoharan et al. 2024](https://arxiv.org/abs/2407.14435)).
- The $\mathbf{d}_i$ are the candidate features; $f_i$ is "how much feature
  $i$ is present". No labels anywhere: unsupervised, which is both the appeal
  and the problem.
- **Loss recovered** judges reconstruction: splice $\hat{\mathbf{x}}$ into the
  model in place of $\mathbf{x}$ and measure how much of the cross-entropy
  increase (relative to zero-ablating the layer) is recovered.
- **$L_0$**, the mean number of active features per token (tens to low
  hundreds), judges sparsity; loss recovered against $L_0$ is the Pareto
  frontier on which SAE architectures are compared.


## Why L1 gives sparsity

![Why the L1 penalty produces exact zeros. (a) In one dimension the penalised optimum is the soft-threshold function: an L2 penalty only shrinks, an L1 penalty sets every coefficient within $\lambda$ of zero to exactly zero. (b) In two dimensions the L1 ball has corners on the axes, and the loss contours meet it there. Source: instructor figure.](figs/23-Interpretability/l1-sparsity.png)

**Quizzable.** Fix every feature but one, with $\|\mathbf{d}_i\|_2 = 1$, and
let $\mathbf{r} = \mathbf{x} - \sum_{j \ne i} f_j \mathbf{d}_j - \mathbf{b}_{\text{dec}}$
be what is left for feature $i$ to explain; the loss in $f_i \ge 0$ is
$\tfrac12 (f_i - a)^2 + \lambda f_i$ up to a constant, with
$a = \mathbf{r}^{\mathsf T}\mathbf{d}_i$ what reconstruction alone wants.
Setting the derivative to zero,
$f_i - a + \lambda = 0 \Rightarrow f_i^* = \max(a - \lambda, 0)$: **every
feature whose unpenalised value is below $\lambda$ is exactly zero**. The L2
penalty $\tfrac{\lambda}{2} f_i^2$ gives $f_i^* = a/(1+\lambda)$, smaller but
never zero. The price of L1 is **shrinkage**, active features under-estimated
by $\lambda$, which is why TopK and JumpReLU, which penalise *count* rather
than *size*, reconstruct better at the same sparsity.


A numerical check of the derivation in its vector form. For a random unit
decoder direction $\mathbf{d}_i$ in 8 dimensions and a residual $\mathbf{r}$
with $a = \mathbf{r}^{\mathsf T}\mathbf{d}_i$ set to a chosen value, a grid
search over $f_i \ge 0$ of the SAE loss
$\tfrac12\|\mathbf{r} - f_i\mathbf{d}_i\|^2 + \lambda f_i\|\mathbf{d}_i\|$ lands
on $\max(a - \lambda, 0)$; the plot shows that solution for three values of
$\lambda$ beside the L2 alternative.

In [ ]:
#| code-fold: true
#| fig-cap: "The optimal non-negative SAE code against what reconstruction alone wants. The L1 penalty zeroes every feature with $a \\le \\lambda$ and shifts the rest down by $\\lambda$; the L2 penalty only scales."
#| fig-alt: "Line chart of optimal code f star against a from minus 1 to 3. A dashed grey line for lambda equal to 0 is max of a and 0. Three solid lines for L1 with lambda 0.5, 1 and 1.5 are zero until a reaches lambda and then rise with slope 1. A dotted black line for L2 with lambda 1 rises from the origin with slope one half."
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
D, lam = 8, 0.5
f_grid = np.linspace(0, 4, 40_001)                 # candidate f_i >= 0, step 1e-4

print(f"lambda = {lam}")
print(f"{'a = r.d':>8} {'grid-search f*':>15} {'max(a - lam, 0)':>16}")
for target in (-1.0, 0.2, 0.5, 0.8, 2.0):
    d = rng.standard_normal(D)
    d /= np.linalg.norm(d)                         # ||d_i|| = 1
    r = rng.standard_normal(D)
    r += (target - r @ d) * d                      # what is left for feature i, with r.d = target
    a = r @ d
    resid = r[None, :] - f_grid[:, None] * d[None, :]
    L = 0.5 * (resid**2).sum(1) + lam * f_grid * np.linalg.norm(d)
    print(f"{a:>8.2f} {f_grid[L.argmin()]:>15.4f} {max(a - lam, 0):>16.4f}")

a = np.linspace(-1, 3, 400)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(a, np.maximum(a, 0), color="gray", ls="--", lw=1, label=r"$\lambda = 0$")
for lam_k in (0.5, 1.0, 1.5):
    ax.plot(a, np.maximum(a - lam_k, 0), lw=2, label=rf"L1, $\lambda = {lam_k}$")
ax.plot(a, np.maximum(a, 0) / 2, color="k", lw=1.2, ls=":", label=r"L2, $\lambda = 1$: $a/2$")
ax.set_xlabel(r"$a = \mathbf{r}^{\top}\mathbf{d}_i$ (what reconstruction alone wants)")
ax.set_ylabel(r"optimal code $f_i^*$")
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Towards Monosemanticity (2023)

- [Bricken et al. (2023)](https://transformer-circuits.pub/2023/monosemantic-features/index.html)
  train SAEs on the 512-neuron MLP of a **one-layer** transformer, with
  dictionaries from 512 up to 131,072 features.
- The features are far more interpretable than the neurons: Arabic script,
  DNA sequences, base64, Hebrew, legal citations, each verified three ways:
  what activates it, what it writes to the logits (its decoder direction
  through the unembedding), and what the model produces when it is
  **clamped** on.
- **Feature splitting.** As the dictionary grows, "punctuation in
  mathematics" splits into finer features. Features are a
  *resolution-dependent* description, not a fixed inventory.
- Automated interpretability: an LLM reads a feature's top examples, writes
  an explanation, and is scored on predicting held-out activations. This is
  how million-feature dictionaries get labelled at all.


## Scaling Monosemanticity: features in a production model

![The Golden Gate Bridge feature (34M/31164353) from Claude 3 Sonnet: it fires on English descriptions of the bridge, on the same concept in Japanese, Korean and Russian, and on photographs of it. Figure from Templeton et al. (2024), "Scaling Monosemanticity", transformer-circuits.pub.](figs/23-Interpretability/golden-gate-feature-templeton.jpg)

- [Templeton et al. (2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)
  train SAEs with about **1M, 4M and 34M features** on a middle-layer
  residual stream of Claude 3 Sonnet. Features are multilingual and
  multimodal (above) and range from concrete to abstract: code bugs,
  sycophantic praise, deception and secrecy, gender bias, AI-risk discussion.
- **Steering demo.** Clamping the Golden Gate feature high makes the model
  bring the bridge into any conversation; clamping a "code error" feature on
  makes it hallucinate a bug, clamping it off on buggy code makes it predict
  the bug-free output. Features are *causal handles*.
- Feature neighbourhoods are organised by meaning, and larger dictionaries
  find rarer concepts, roughly in Zipf order.
- Caveats the paper states: the SAE explains only part of the variance, many
  features are uninterpreted, and finding *which* features matter for a
  behaviour still needs attribution or ablation.


## Open SAE suites you can download

- **Gemma Scope** ([Lieberum et al. 2024](https://arxiv.org/abs/2408.05147)):
  more than 400 JumpReLU SAEs, over 30M features, on every layer and
  sub-layer of Gemma 2 2B and 9B (some of 27B), each trained on 4 to 16
  billion tokens, with [Neuronpedia](https://www.neuronpedia.org/) pages for
  every feature.
- **Gemma Scope 2** (released mid-December 2025;
  [McDougall et al. 2025](https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/gemma-scope-2-helping-the-ai-safety-community-deepen-understanding-of-complex-language-model-behavior/Gemma_Scope_2_Technical_Paper.pdf),
  [`google/gemma-scope-2`](https://huggingface.co/google/gemma-scope-2) on
  Hugging Face, CC-BY-4.0): SAEs, skip-transcoders, crosscoders and CLTs for
  Gemma 3 270M–27B, pre-trained and instruction-tuned, widths 16k–1M, target
  $L_0$ from 10 to 150; the transcoders and crosscoders are for circuit work
  (section 5), and SAEs fine-tuned on chat data are for studying the assistant.
- **16M latents:** [Gao et al. (2024)](https://arxiv.org/abs/2406.04093)
  train a TopK SAE with 16 million latents on GPT-4 activations over 40B
  tokens and report clean scaling laws against dictionary size and sparsity.
- **Reasoning models:** [Goodfire (2025)](https://www.goodfire.ai/blog/under-the-hood-of-a-reasoning-model)
  released SAEs trained on DeepSeek-R1 (671B), with features for behaviours
  such as backtracking in a chain of thought.
- All of it runs on one GPU with the base model in bf16; the "Hands-on
  tools" slide has the code.


## SAE limitations and the 2025 reality check

- **Feature splitting and absorption.** A general feature ("starts with S")
  can be *absorbed* into specific ones ("short", "snake") and silently stop
  firing where it should ([Chanin et al. 2024](https://arxiv.org/abs/2409.14507)).
  Which concepts get a direction is an artefact of dictionary size and training.
- **Not the model's own basis.** An SAE is a hypothesis fitted to a dataset;
  another seed gives a partly different dictionary; features can be
  compositions or fragments of the "true" features, if those exist.
- **Downstream tasks.** [Kantamneni et al. (2025)](https://arxiv.org/abs/2502.16681)
  (ICML 2025) test SAE probes under data scarcity, class imbalance, label
  noise and covariate shift and cannot make them consistently beat linear
  probes on raw activations. Google DeepMind's team reported the same for
  out-of-distribution detection of harmful intent and announced it was
  **deprioritising** fundamental SAE research
  ([Smith et al. 2025](https://www.alignmentforum.org/posts/4uXCAJNuPKtKBsi28/sae-progress-update-2-negative-results-for-saes-on-downstream)).
- Balanced reading: SAEs are a strong **exploratory microscope** and a weak
  **classifier**. The 2025 pivot was to use dictionaries to build *graphs of
  computation*, the next section.


# Circuit tracing and attribution graphs

A dictionary tells you which features are present on a token, not which
feature caused which. The 2025 work from Anthropic joins the two halves of
this lecture, dictionary features and causal tracing, into a graph of the
computation for one prompt, and used it to find mechanisms in a production
model.


## From features to computation: transcoders

- An SAE reconstructs an activation from itself. A **transcoder** learns a
  sparse, interpretable *replacement for an MLP*: it reads the MLP's input and
  predicts the MLP's output through a wide sparse bottleneck
  ([Dunefsky et al. 2024](https://arxiv.org/abs/2406.11944)). A feature now
  has an *input side* (encoder) and an *output side* (decoder), so
  feature-to-feature effects are linear and computable.
- **Cross-layer transcoders (CLTs)**
  ([Ameisen et al. 2025](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)):
  each feature reads the stream at its layer and writes to the MLP outputs of
  *every later layer*, absorbing features the model smears across layers.
  Loss: reconstruction over all layers plus a sparsity penalty,
  $L = \sum_\ell \|\hat{\mathbf{y}}^\ell - \mathbf{y}^\ell\|^2 + \lambda \sum_{\ell, i} \tanh\big(c\,\|\mathbf{d}_i^\ell\|\, f_i^\ell\big)$.
- **Replacement model.** Swap every MLP for its CLT and freeze the attention
  patterns and normalisation denominators at their values on the prompt. The
  result is *linear in the features* for that prompt; its next token matches
  the real model's on about half of diverse prompts for the largest 18-layer
  research model, and the mismatch is carried by explicit **error nodes**.
- Gemma Scope 2 ships transcoders and CLTs for Gemma 3, so this can be done
  on an open model.


## Attribution graphs

- For one prompt, build a directed graph: nodes are the **active features**,
  the input **token embeddings**, the **error nodes** and the **output
  logits**; an edge from feature $s$ to feature $t$ is the **direct linear
  effect** of $s$'s activation on $t$'s pre-activation through the frozen
  attention and the decoder-to-encoder path, $A_{s \to t} = f_s \cdot w_{s \to t}$.
- The raw graph has thousands of nodes. **Prune** to those carrying most of
  the influence on the output: roughly 10× fewer nodes keeping about 80% of
  the explanatory value. Group similar features into **supernodes** labelled
  by their top examples.
- **Validate by intervention**, as in section 3: suppress or amplify a
  feature in the *real* model and check that downstream features and the
  output move as the graph predicts. A graph is a hypothesis; patching is
  the test.
- Not in the graph: *why the attention patterns are what they are* (the QK
  circuits are frozen), and whatever sits in the error nodes.


## Case study: two-hop reasoning inside one forward pass

![The pruned attribution graph for "the capital of the state containing Dallas is": Dallas and state features activate a Texas feature, which with a "say a capital" feature activates "say Austin". Figure from Lindsey et al. (2025), "On the Biology of a Large Language Model", transformer-circuits.pub.](figs/23-Interpretability/attribution-graph-dallas-lindsey.png)

- [Lindsey et al. (2025)](https://transformer-circuits.pub/2025/attribution-graphs/biology.html)
  apply the method to **Claude 3.5 Haiku**. The graph shows an intermediate
  **Texas** representation that is never output: the model does the two hops
  internally rather than recalling a memorised "Dallas → Austin" pair.
- Intervention: suppress the Texas features and inject **California**
  features, and the model answers **Sacramento**; inject other states and it
  answers their capitals. The same "say a capital" machinery is reused, which
  is what a general mechanism looks like.
- Evals cannot settle this (a lookup table and a two-step reasoner both get
  it right) and a probe cannot either (a Texas probe fires in both cases).
  Only the intervention distinguishes them.


## Case study: planning in poems

![When writing a rhyming couplet, features for candidate rhyme words ("rabbit", "habit") are active at the newline *before* the second line is written, driven by a "rhymes with -it" feature. Figure from Lindsey et al. (2025), transformer-circuits.pub.](figs/23-Interpretability/attribution-graph-poem-lindsey.png)

- The prompt gives the first line of a couplet ending in "grab it". At the
  newline the model already carries features for **rabbit** and **habit**,
  candidate end-words for a line it has not started; the line is then
  written *towards* the chosen word.
- Intervention: suppress "rabbit" and the line ends on "habit"; inject an
  unrelated word ("green") and the model writes a sensible line ending in
  "green". The planned word is causally upstream of the whole line. A
  next-token predictor is not forbidden from planning.
- Also in the paper: a language-independent core with language-specific
  input/output features; addition by parallel approximate and lookup-table
  features; a **"known entity"** feature that inhibits a default "can't
  answer" circuit, whose misfiring is one mechanism of hallucination; a
  jailbreak that works because the model cannot see what it is spelling
  until it has said it; and chain-of-thought steps whose stated arithmetic
  is not the arithmetic the graph shows.


## What was learned, and what is still unexplained

- **Learned.** Production models contain reusable intermediate concepts,
  multi-step reasoning in one forward pass, forward planning, and circuits
  for refusal and for "do I know this?": facts about a specific model,
  verified by intervention.
- **Limits the authors state.** Satisfying insight on about a quarter of the
  prompts tried; graphs are per-prompt and give no global mechanism by
  themselves; attention-pattern formation is outside the method; error nodes
  and uninterpreted features carry an unknown share; every published case is
  a success case.
- **Tooling.** [`circuit-tracer`](https://github.com/safety-research/circuit-tracer)
  (May 2025) builds attribution graphs for open-weight models with Gemma
  Scope transcoders, and [Neuronpedia](https://www.neuronpedia.org/) hosts an
  interactive viewer. A graph is a hypothesis generator: read it, guess the
  mechanism, then test with patching and steering on the real model.


# Steering and editing

If a concept is a direction, you can add it. If a fact lives in an MLP, you can
overwrite it. This section covers the two interventions that follow from the
representational story, their successes and the failure modes that are easy to
miss, then the use of all of this as a safety audit.


## Activation steering

- **Add a difference-of-means vector.** Record the stream at layer $l$ on
  prompts with and without the property ("Love" / "Hate", sycophantic / not)
  and add the scaled difference during generation:
  $$
  \mathbf{h}_l \leftarrow \mathbf{h}_l + c\,\big(\bar{\mathbf{h}}_l^{+} - \bar{\mathbf{h}}_l^{-}\big).
  $$
  One pair in activation addition ([Turner et al. 2023](https://arxiv.org/abs/2308.10248)),
  many pairs in contrastive activation addition
  ([Rimsky et al. 2023](https://arxiv.org/abs/2312.06681)), a PCA direction
  that doubles as a monitor in representation engineering
  ([Zou et al. 2023](https://arxiv.org/abs/2310.01405)). No optimisation.
- **One direction for refusal** ([Arditi et al. 2024](https://arxiv.org/abs/2406.11717)):
  in 13 open chat models a single direction mediates refusal; subtracting it
  jailbreaks with no fine-tuning, adding it makes the model refuse harmless
  requests. **Persona vectors** ([Chen et al. 2025](https://arxiv.org/abs/2507.21509))
  do the same for traits (sycophancy, hallucination) and can stop a trait
  drifting during fine-tuning.


## SAE-feature steering, model editing, and how they fail

- **Steering with a dictionary feature** is activation addition along
  $\mathbf{d}_i$ (Golden Gate Claude; the code-error feature). The feature
  comes with a label and examples; but strong clamping degrades everything
  else, and a feature is not guaranteed to be the model's own handle.
- **ROME** ([Meng et al. 2022](https://arxiv.org/abs/2202.05262)): treat a
  mid-layer MLP as a key-value memory,
  $\mathbf{v} = \boldsymbol{\Omega}_2\,\mathrm{a}[\boldsymbol{\Omega}_1\mathbf{h}]$,
  and write a new fact with the update
  $\boldsymbol{\Omega}_2 \leftarrow \boldsymbol{\Omega}_2 + \mathbf{u}\,(\mathbf{C}^{-1}\mathbf{k}_*)^{\mathsf T}$,
  with $\mathbf{k}_*$ the subject's key, $\mathbf{C}$ the key covariance (so
  $\mathbf{C}^{-1}\mathbf{k}_*$ is a whitened key) and $\mathbf{u}$ chosen so
  the output becomes the new value ("Eiffel Tower is in → Rome"); up to that
  whitening it is a **rank-one update**.
- **Failure modes.** Edits **bleed** into unrelated prompts that share tokens
  with the subject, **ripple effects** are missing (after "Eiffel Tower →
  Rome" the model still takes the Paris Métro to it), and the layer causal
  tracing *localises* a fact to is not reliably the best layer to *edit*, so
  localisation and editability are different questions
  ([Hoelscher-Obermaier et al. 2023](https://arxiv.org/abs/2305.17553),
  [Cohen et al. 2023](https://arxiv.org/abs/2307.12976),
  [Hase et al. 2023](https://arxiv.org/abs/2301.04213)).
- The moral from section 3 again: a representation being *present* and
  *used* at one site does not make that site the only one, nor the one a
  gradient step would pick.


## Interpretability as a safety audit

- The safety case: a model may have a goal it does not state. Behavioural
  evals test the behaviours you thought of; an audit of the internals might
  catch one you did not.
- [Marks et al. (2025)](https://arxiv.org/abs/2503.10965) ran the
  experiment: a model trained with a **hidden objective** (exploit known
  biases of RLHF reward models, "RM-sycophancy") via synthetic documents
  describing the biases and fine-tuning on a subset; it generalised to biases
  it was never trained to exploit. Four teams who did not know the objective
  audited it blind.
- **Result:** three of four teams found it. The three successful teams
  (Teams 1–3) had the weights and the training data; one of them solved it
  with black-box prefill attacks plus keyword search over the data. What
  worked: SAE features (one firing on discussions of reward-model biases led
  to the synthetic documents), behavioural attacks (making the model play the
  *user*, who was never trained to keep the secret), and training-data search.
  Team 4, non-Anthropic researchers with API access only (no weights, no
  data), failed (Marks et al. §4.2, Table 1).
- Lindsey et al. (2025) traced the same model's goal to features in its
  representation of its own **assistant persona**. The caveat: a planted
  objective of known shape, audited by the people who built the tools.


# Honest limits and open problems

The field's results are real and its claims are usually careful. This section
collects the limits that matter for a practitioner: the model's own
explanations cannot be trusted by default, features are not understanding, and
the tools are easier to run than the conclusions are to draw.


## Is the chain of thought faithful?

- The cheapest "interpretability" is to ask the model why. **It does not
  always say what it thinks.** [Turpin et al. (2023)](https://arxiv.org/abs/2305.04388):
  reorder few-shot examples so the correct answer is always (A), and GPT-3.5
  and Claude 1.0 drop up to 36 points on BIG-Bench Hard tasks while writing
  chains of thought that *rationalise* the biased answer and never mention
  the pattern.
- [Lanham et al. (2023)](https://arxiv.org/abs/2307.13702) truncate or
  corrupt the chain and check whether the answer changes; on many tasks it
  does not, and larger models' chains are *less* load-bearing on easy tasks.
- Reasoning models ([Chen et al. 2025](https://arxiv.org/abs/2505.05410)):
  give Claude 3.7 Sonnet and DeepSeek-R1 a hint, and when the model uses it
  the chain **mentions the hint, in most settings, less than 20% of the
  time**; reward hacks learned in RL are exploited almost without ever being
  verbalised.
- Monitoring the chain still catches a lot ([Baker et al. 2025](https://arxiv.org/abs/2503.11926)),
  but **optimising against the monitor** teaches the model to hide intent in
  an innocuous chain, so [Korbak et al. (2025)](https://arxiv.org/abs/2507.11473)
  argue monitorability is fragile and must be preserved, not assumed. The
  chain of thought is an *output*; section 5's graphs showed its stated steps
  can differ from the computed ones.


## "Features exist" is not "we understand the model"

- Section 5's limits generalise: millions of labelled directions and
  per-prompt graphs are not a global description of **what a production
  model computes**, nor a guarantee that a behaviour absent from our prompts
  is absent from the model.
- **Open problems** ([Sharkey et al. 2025](https://arxiv.org/abs/2501.16496)):
  are features the right primitive, or should we decompose *parameters*? How
  do we evaluate an explanation without ground truth? Can this scale to the
  largest models at a fraction of training cost? How does a finding become a
  *guarantee* an engineer can use?
- **Two cheap errors.** Reading a feature's label as its meaning (labels are
  LLM summaries of top examples); and reading a patching result as "this is
  where X happens" (it is where X was *sufficient to restore* the behaviour
  under *this* corruption).


## Hands-on tools on one GPU

[TransformerLens](https://github.com/TransformerLensOrg/TransformerLens)
(hooked forward passes, patching), [SAELens](https://github.com/jbloomAus/SAELens)
(train and load SAEs, including Gemma Scope), [Neuronpedia](https://www.neuronpedia.org/)
(feature pages, steering, attribution graphs in the browser),
[`circuit-tracer`](https://github.com/safety-research/circuit-tracer).

```python
from transformer_lens import HookedTransformer
from sae_lens import SAE

model = HookedTransformer.from_pretrained("gemma-2-2b")          # open weights, bf16 fits on one GPU
tokens = model.to_tokens("The capital of the state containing Dallas is")
logits, cache = model.run_with_cache(tokens)

# Logit lens: unembed the residual stream after every block at the last position.
for layer in range(model.cfg.n_layers):
    h = cache["resid_post", layer][0, -1]
    top = model.unembed(model.ln_final(h[None, None]))[0, -1].argmax()
    print(layer, model.to_string(top))

# Activation patching: overwrite layer 12's residual stream at the city position.
corrupt = model.to_tokens("The capital of the state containing Oakland is")
def patch(resid, hook, pos=7):
    resid[0, pos] = cache["resid_post", 12][0, pos]        # clean value in, corrupted run
    return resid
patched = model.run_with_hooks(corrupt, fwd_hooks=[("blocks.12.hook_resid_post", patch)])

# Sparse autoencoder features (Gemma Scope) on the same activations.
sae = SAE.from_pretrained("gemma-scope-2b-pt-res", "layer_12/width_16k/average_l0_82")
# SAELens >= 6; older versions returned (sae, cfg, sparsity)
feats = sae.encode(cache["resid_post", 12])                      # [batch, pos, 16384], mostly zero
print((feats[0, -1] > 0).sum(), feats[0, -1].topk(5))            # L0 and the top features
```

An afternoon's project: pick one factual prompt, run the logit lens, run a
residual-stream patching sweep, and look up the top five SAE features at the
layer the sweep points to on Neuronpedia. You will have reproduced the
structure of sections 3 and 4 on a 2B model.


# Summary

One line per idea; the recap deck uses these.


## Summary

- Evals measure behaviour on a distribution; interpretability asks why, whether it transfers, and what has not shown up yet. Two traditions: attribution (which inputs) and mechanistic (which internal computation).
- Plain gradients fail under saturation and fail sanity checks; integrated gradients $\mathrm{IG}_i = (x_i - x_i')\int_0^1 \partial_i F[\mathbf{x}' + \alpha(\mathbf{x}-\mathbf{x}')]\,d\alpha$ satisfy sensitivity, implementation invariance and completeness ($\sum_i \mathrm{IG}_i = F[\mathbf{x}] - F[\mathbf{x}']$); Grad-CAM is the CNN-era analogue. Attribution says where, not how.
- The residual stream is an additive bus with no privileged basis; each head is a QK circuit $\boldsymbol{\Omega}_k^{\mathsf T}\boldsymbol{\Omega}_q$ (where to attend) and an OV circuit $\boldsymbol{\Omega}_o\boldsymbol{\Omega}_v$ (what to copy); heads compose through virtual weights.
- The logit lens unembeds the stream at any depth. Induction heads (previous-token head + K-composition) implement `[A][B]...[A] → [B]`, form in a phase change, and drive in-context learning.
- Linear representation hypothesis: concepts are directions. Linear probes show information is present at a layer (use control tasks); Othello-GPT's board is linear in the mine/theirs basis. A probe cannot show the model uses the information.
- Activation patching is the causal test: effect $= (\Delta_{\text{patched}} - \Delta_{\text{corrupted}})/(\Delta_{\text{clean}} - \Delta_{\text{corrupted}})$ on a logit difference. Causal tracing locates factual recall in early-middle MLPs at the last subject token.
- Superposition: sparse features let a model store more features than dimensions as nearly-orthogonal directions, which makes neurons polysemantic.
- A sparse autoencoder learns an over-complete dictionary with $L = \tfrac12\|\mathbf{x} - \hat{\mathbf{x}}\|^2 + \lambda\sum_i f_i\|\mathbf{d}_i\|$; L1 gives exact zeros by soft-thresholding, $f_i^* = \max(a - \lambda, 0)$, at the cost of shrinkage; TopK and JumpReLU penalise count instead.
- Scaling Monosemanticity found 34M features in Claude 3 Sonnet, multilingual, multimodal and steerable; Gemma Scope and Gemma Scope 2 give open SAEs and transcoders for Gemma 2 and 3. SAEs split and absorb features and did not beat linear probes on downstream tasks in 2025.
- Cross-layer transcoders replace MLPs with sparse features; attribution graphs link features by direct linear effects for one prompt, validated by intervention; they showed two-hop reasoning, planning in poems and unfaithful chain-of-thought steps, with insight on about a quarter of prompts tried.
- Steering adds a direction: a difference-of-means vector (activation addition, contrastive activation addition, representation engineering), a single refusal direction, a persona vector, an SAE feature. ROME edits a fact with a rank-one MLP update; edits bleed, do not ripple, and localisation does not imply editability.
- A blind audit found a planted hidden objective in three of four teams; the chain of thought is unfaithful often enough (hints verbalised under 20% in most settings) that it is an output to monitor, not an explanation to trust.
- Finding features is not understanding the model; open problems are global descriptions, evaluation without ground truth, and turning findings into guarantees. TransformerLens, SAELens, Neuronpedia and circuit-tracer run on one GPU.

## Discussion questions

1. A probe for "the current answer is false" reaches 95% accuracy at layer 20 of an open model. Design the cheapest experiment that would tell you whether the model *uses* that direction, and say what result would make you stop trusting the probe.
2. The SAE loss uses L1 on the features. Say what happens to the dictionary if you replace it with L2, and why TopK avoids the shrinkage problem but introduces a different failure (think about tokens that genuinely need more than $k$ features).
3. Lindsey et al. report insight on about a quarter of prompts. Is that a reason to distrust the quarter that worked? What would convince you that a found mechanism is *the* mechanism, not one of several?
4. Section 7 says the chain of thought is an output, not an explanation. Lecture 21 built agents that act on their chain of thought. Which of this lecture's tools would you add to an agent harness as a monitor, and what would it cost per step?

## Further reading

- [Elhage et al., *A Mathematical Framework for Transformer Circuits* (2021)](https://transformer-circuits.pub/2021/framework/index.html): the residual-stream view, QK/OV circuits, and induction heads in two-layer models. Read sections 1 to 3.
- [Olsson et al., *In-context Learning and Induction Heads* (2022)](https://arxiv.org/abs/2209.11895): the phase change and the six arguments.
- [Elhage et al., *Toy Models of Superposition* (2022)](https://arxiv.org/abs/2209.10652): the figure on the superposition slide and the geometry behind it.
- [Templeton et al., *Scaling Monosemanticity* (2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) and [Lieberum et al., *Gemma Scope* (2024)](https://arxiv.org/abs/2408.05147): SAEs on a production model, and the open suite you can run.
- [Ameisen et al., *Circuit Tracing* (2025)](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) and [Lindsey et al., *On the Biology of a Large Language Model* (2025)](https://transformer-circuits.pub/2025/attribution-graphs/biology.html): the method and the case studies; the biology paper's limitations section is the best short statement of where the field stands.
- [Sharkey et al., *Open Problems in Mechanistic Interpretability* (2025)](https://arxiv.org/abs/2501.16496): the agenda.
- [Neel Nanda's *Mechanistic Interpretability Quickstart Guide*](https://www.neelnanda.io/mechanistic-interpretability/quickstart) and the [ARENA](https://www.arena.education/) interpretability track: exercises that teach TransformerLens, patching and SAEs in a week.